### How to filter and view data using mlflow CLI 

In [52]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [53]:
experiments = client.search_experiments()

# for exp in experiments:
#     print(f"id={exp.experiment_id}, name={exp.name}, stage={exp.lifecycle_stage}")

experiments

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp-megh/02-experiment-tracking/mlruns/1', creation_time=1780564663873, experiment_id='1', last_update_time=1780564663873, lifecycle_stage='active', name='green_tripdata_2026-01-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1780564643959, experiment_id='0', last_update_time=1780564643959, lifecycle_stage='active', name='Default', tags={}>]

In [54]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="params.model_type = 'XGBoost Regression'",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=10,
    order_by=["metrics.rmse ASC"]
)

In [55]:
for run in runs:
    rmse = run.data.metrics.get("rmse")
    if rmse is None:
        print(f"run_id={run.info.run_id}, rmse missing")
        continue
    print(f"run_id={run.info.run_id}, rmse={rmse}")

run_id=bf96255863124f8eb9efa87f1ae8106b, rmse=0.1523692790272557
run_id=2cbdc1a4088f429cb657ea832f988187, rmse=0.1523692790272557
run_id=c01d5a647a514c47b731d76c3e4a0e04, rmse=0.1523692790272557
run_id=0b7e03e1d6584e2db9ad30540391f3a6, rmse=0.1523692790272557
run_id=7c6617b46da641909bdd101c42cfda13, rmse missing


### How to use model registry features in mlflow CLI

In [56]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [57]:
run_id = "2cbdc1a4088f429cb657ea832f988187"
model_uri = f"runs:/{run_id}/model"

mlflow.register_model(
    model_uri=model_uri,
    name="nyc-taxi-duration-prediction-model"
)

Registered model 'nyc-taxi-duration-prediction-model' already exists. Creating a new version of this model...
2026/06/08 13:32:57 WARNING mlflow.tracking._model_registry.fluent: Run with id 2cbdc1a4088f429cb657ea832f988187 has no artifacts at artifact path 'model', registering model based on models:/m-000de5efa3b147e0bea8c85ddc89a739 instead
Created version '6' of model 'nyc-taxi-duration-prediction-model'.


<ModelVersion: aliases=[], creation_timestamp=1780925577233, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1780925577233, metrics=None, model_id=None, name='nyc-taxi-duration-prediction-model', params=None, run_id='2cbdc1a4088f429cb657ea832f988187', run_link=None, source='models:/m-000de5efa3b147e0bea8c85ddc89a739', status='READY', status_message=None, tags={}, user_id=None, version=6>

### Transition a model from stages

In [58]:
for version in client.search_model_versions("name='nyc-taxi-duration-prediction-model'"):
    print(f"version={version.version}, run_id={version.run_id}, current_stage={version.current_stage}")

version=6, run_id=2cbdc1a4088f429cb657ea832f988187, current_stage=None
version=1, run_id=, current_stage=Production
version=5, run_id=, current_stage=None
version=4, run_id=2cbdc1a4088f429cb657ea832f988187, current_stage=None
version=3, run_id=, current_stage=Staging


In [59]:
client.transition_model_version_stage(
    name="nyc-taxi-duration-prediction-model",
    version=1,
    stage="Production"
)

/tmp/ipykernel_16376/4271957751.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1780899091560, current_stage='Production', deployment_job_state=None, description='The model version 1 was transitioned to Production stage on 2023-01-01.', last_updated_timestamp=1780925577423, metrics=None, model_id=None, name='nyc-taxi-duration-prediction-model', params=None, run_id='', run_link='', source='/workspaces/mlops-zoomcamp-megh/02-experiment-tracking/mlruns/1/models/m-e2d6a9fd50a045dd8977cdda35b31922/artifacts', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=1>

In [60]:
client.update_model_version(
    name="nyc-taxi-duration-prediction-model",
    version=1,
    description="The model version {version} was transitioned to {stage} stage on {date}."
)

<ModelVersion: aliases=[], creation_timestamp=1780899091560, current_stage='Production', deployment_job_state=None, description='The model version {version} was transitioned to {stage} stage on {date}.', last_updated_timestamp=1780925577516, metrics=None, model_id=None, name='nyc-taxi-duration-prediction-model', params=None, run_id='', run_link='', source='/workspaces/mlops-zoomcamp-megh/02-experiment-tracking/mlruns/1/models/m-e2d6a9fd50a045dd8977cdda35b31922/artifacts', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=1>

### Testing a registered model against the dataset

In [61]:
from sklearn.metrics import mean_squared_error
import pandas as pd

def read_dataframe(filename) -> pd.DataFrame:
    df = pd.read_parquet(filename)

    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda x: x.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df

def preprocess(df, dv) -> pd.DataFrame:
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numeric = ['trip_distance']

    dicts = df[categorical].to_dict(orient='records')
    X = dv.transform(dicts)

    return X

def test_model(name="nyc-taxi-duration-prediction-model", stage=None, X_test=None, y_test=None) -> float:
    model = mlflow.pyfunc.load_model(model_uri=f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    return rmse

In [62]:
df = read_dataframe("../data/green_tripdata_2025-02.parquet")

In [64]:
run_id = "bf96255863124f8eb9efa87f1ae8106b"
client.download_artifacts(run_id, path="preprocessor", dst_path=".")

'/workspaces/mlops-zoomcamp-megh/02-experiment-tracking/preprocessor'

In [66]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [67]:
X_test = preprocess(df, dv)

In [68]:
target = 'duration'
y_test = df[target].values

In [70]:
%time rmse = test_model(stage="Staging", X_test=X_test, y_test=y_test)
print(f"RMSE: {rmse}")

CPU times: user 4.57 s, sys: 94.5 ms, total: 4.67 s
Wall time: 4.83 s
RMSE: 45.50401257665273


In [72]:
client.transition_model_version_stage(
    name="nyc-taxi-duration-prediction-model",
    version=4,
    stage="Production",
    archive_existing_versions=True
)

/tmp/ipykernel_16376/434943020.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1780923308419, current_stage='Production', deployment_job_state=None, description=None, last_updated_timestamp=1780925827927, metrics=None, model_id=None, name='nyc-taxi-duration-prediction-model', params=None, run_id='2cbdc1a4088f429cb657ea832f988187', run_link=None, source='models:/m-000de5efa3b147e0bea8c85ddc89a739', status='READY', status_message=None, tags={}, user_id=None, version=4>